# 05 · Gemma 3n E2B challenger: LoRA SFT + val comparison (A100)
Same data, prompt, schema, augmentation and optimiser settings as the SmolVLM-500M runs (`configs/train_gemma3n_e2b.yaml`), so differences come from the base model.
**Val only.** This notebook never touches `--split test`. If Gemma wins on val, update `reports/preregistration.json` (notebook 04) *before* the locked test.
Runtime → Change runtime type → **A100 GPU** (L4 works with smaller batches + gradient checkpointing).

## 0. Session setup (run every session)
Code is cloned from GitHub (`moghbaie/InspectEdge_Mobile_VLM`); nothing needs to be uploaded.
Google Drive (`MyDrive/InspectEdge`) only stores outputs: reports, manifests, checkpoints.
Runtime → Change runtime type → **GPU** (T4 is enough for 500M; L4/A100 for 3B).

In [ ]:
import os, json, subprocess
from google.colab import drive
drive.mount('/content/drive')
REPO  = 'https://github.com/moghbaie/InspectEdge_Mobile_VLM.git'
DRIVE = '/content/drive/MyDrive/InspectEdge'   # persistent outputs: reports, manifests, checkpoints
WORK  = '/content/InspectEdge'                 # fast local disk: code + raw images
os.makedirs(DRIVE, exist_ok=True)
if os.path.isdir(f'{WORK}/.git'):
    !git -C {WORK} pull -q --ff-only
else:
    !git clone -q {REPO} {WORK}
assert os.path.exists(f'{WORK}/pyproject.toml'), 'clone failed: check REPO URL / repo visibility'
%cd {WORK}
!git log --oneline -1
!pip -q install -r requirements-colab.txt
!pip -q install -e . --no-deps
os.environ['INSPECTEDGE_ROOT'] = WORK
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv
import torch, transformers, peft
assert transformers.__version__ == '4.57.1', 'pinned transformers not active: Runtime -> Restart session, then re-run this cell'
print('torch', torch.__version__, 'cuda', torch.version.cuda, '| transformers', transformers.__version__, '| peft', peft.__version__)
assert torch.cuda.is_available(), 'No GPU: Runtime -> Change runtime type -> GPU'

In [ ]:
# Data: official VisA release (CC BY 4.0). Verified by SHA-256 before use.
VISA_SHA = '2eb8690c803ab37de0324772964100169ec8ba1fa3f7e94291c9ca673f40f362'
import pandas as pd
def visa_missing():   # every manifest image/mask must be on local disk (a runtime reset or partial extract loses them)
    man = pd.read_json('data/manifests/visa_manifest.jsonl', lines=True)
    return [p for p in list(man.image_path) + [m for m in man.mask_path if m] if not os.path.exists(p)]
if visa_missing():
    !rm -rf data/raw/visa data/raw/_dl
    os.makedirs('data/raw/_dl', exist_ok=True); os.makedirs('data/raw/visa', exist_ok=True)
    !wget -q -O data/raw/_dl/VisA_20220922.tar https://amazon-visual-anomaly.s3.us-west-2.amazonaws.com/VisA_20220922.tar
    h = subprocess.run(['sha256sum', 'data/raw/_dl/VisA_20220922.tar'], capture_output=True, text=True).stdout.split()[0]
    assert h == VISA_SHA, f'VisA archive checksum mismatch: {h}'
    !tar -xf data/raw/_dl/VisA_20220922.tar -C data/raw/visa && rm data/raw/_dl/VisA_20220922.tar
_miss = visa_missing()
assert not _miss, f'{len(_miss)} VisA files missing after extraction (disk full?), e.g. {_miss[0]}'
open('data/raw/visa/.complete', 'w').close()
!ls data/raw/visa

In [ ]:
# Manifest: rebuilt deterministically each session; fails loudly if the locked split changed.
!python -m inspectedge.data.build_manifest

In [ ]:
# Ampere+ GPU (A100 preferred; L4/H100 also fine). bf16 + TF32 need compute capability >= 8.0.
gpu = torch.cuda.get_device_name(0); mem_gb = torch.cuda.get_device_properties(0).total_memory / 2**30
assert torch.cuda.is_bf16_supported() and torch.cuda.get_device_capability(0)[0] >= 8, \
    f'{gpu} lacks bf16: Runtime -> Change runtime type -> A100 (or L4) GPU'
torch.backends.cuda.matmul.allow_tf32 = True; torch.backends.cudnn.allow_tf32 = True
print(gpu, f'{mem_gb:.0f} GB')

## Hugging Face access (Gemma is gated)
1. Accept the Gemma terms at https://huggingface.co/google/gemma-3n-E2B-it.
2. Add a read token as a Colab secret named `HF_TOKEN` (🔑 icon in the left sidebar) and enable notebook access.

In [ ]:
from google.colab import userdata
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
!pip -q install "timm>=1.0.16"   # Gemma 3n vision tower (MobileNet-V5) is a timm model
from huggingface_hub import hf_hub_download
MODEL = 'google/gemma-3n-E2B-it'
hf_hub_download(MODEL, 'config.json')   # fails here if the license is not accepted / token missing
print('access OK:', MODEL)

In [ ]:
# Memory-sized overrides; effective batch stays 16 so lr/warmup match the SmolVLM runs.
# Gemma 3n is ~5B raw params with a 262k vocab (large logits), so batches are smaller than SmolVLM's.
if mem_gb > 60:   BS, CKPT_ = 8, False   # A100/H100 80GB
elif mem_gb > 30: BS, CKPT_ = 4, True    # A100 40GB
else:             BS, CKPT_ = 2, True    # L4 24GB
GPU_SET = f'batch_size={BS} grad_accum={16 // BS} grad_ckpt={str(CKPT_).lower()} workers={min(8, os.cpu_count())} tf32=true'
CFG = 'configs/train_gemma3n_e2b.yaml'
RUN = 'gemma3n_e2b_lora_r16_lr2e-4_s0'
CKPT = f'{DRIVE}/checkpoints'
print(GPU_SET)

## 1. Zero-shot B2 reference (same 10-per-cell val sample as the SmolVLM B2 baseline)

In [ ]:
!python scripts/eval_vlm.py --model {MODEL} --split val --per-cell 10 2>&1 | grep -vE 'Warning|warn' | tail -25

## 2. One-batch overfit smoke test (must print SMOKE TEST PASSED)
Written to `checkpoints/smoke` (shared with notebook 03; it is only a pass/fail check).

In [ ]:
!python scripts/train_vlm_lora.py --config {CFG} --smoke --set output_dir={CKPT} {GPU_SET} 2>&1 | grep -vE 'Warning|warn' | tail -15

## 3. Main run (resumable: re-run this cell after a disconnect; on OOM halve `batch_size`, double `grad_accum`, set `grad_ckpt=true`)

In [ ]:
!python scripts/train_vlm_lora.py --config {CFG} --resume --set output_dir={CKPT} {GPU_SET} 2>&1 | grep -vE 'Warning|warn' | tail -30

In [ ]:
!nvidia-smi --query-gpu=memory.used,memory.total --format=csv

In [ ]:
meta = json.load(open(f'{CKPT}/{RUN}/run_meta.json'))
import matplotlib.pyplot as plt; plt.plot(meta['losses']); plt.title(f'{RUN} train loss'); plt.show()
print(meta['gpu'], round(meta['train_seconds']/60,1), 'min')

## 4. Full-val evaluation of the fine-tuned adapter

In [ ]:
!python scripts/eval_vlm.py --model {MODEL} --adapter {CKPT}/{RUN}/final --split val --per-cell 0 --tag {RUN} 2>&1 | grep -vE 'Warning|warn' | tail -22

## 5. Val comparison with the SmolVLM runs (metrics from this session + those synced to Drive by notebook 04)

In [ ]:
import glob, pandas as pd
KEYS = ['schema_valid_rate','status_acc_strict','anomaly_balanced_acc','anomaly_recall_flagged','false_reassurance_rate',
        'unsupported_finding_rate_on_normals','defect_type_macro_f1','region_iou>=0.3_rate','latency_p50_s','latency_p95_s','peak_gpu_mem_gb']
rows = {}
for f in sorted(glob.glob(f'{DRIVE}/reports/vlm_eval/*_val_metrics.json')) + sorted(glob.glob('reports/vlm_eval/*_val_metrics.json')):
    m = json.load(open(f))
    if m.get('corrupt'): continue
    rows[os.path.basename(f).replace('_val_metrics.json', '')] = {k: m.get(k) for k in KEYS}   # local (newer) wins
df = pd.DataFrame(rows).T.sort_values('status_acc_strict', ascending=False)
gates = {'schema_valid_rate': ('>=', .995), 'anomaly_recall_flagged': ('>=', .85),
         'unsupported_finding_rate_on_normals': ('<=', .10), 'false_reassurance_rate': ('<=', .15)}
df['gates_pass'] = df.apply(lambda r: all((r[k] >= t) if op == '>=' else (r[k] <= t) for k, (op, t) in gates.items()), axis=1)
df.round(3)

Zero-shot rows (no `lora` in the name) use a 10-per-cell sample, fine-tuned rows the full val split, so compare fine-tuned rows with each other. Latency is server-GPU latency; compare only rows measured on the same GPU type.

In [ ]:
# Persist outputs to Drive (run at the end and after long steps)
!rsync -aR reports data/manifests {DRIVE}/
print('synced to', DRIVE)